# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdarshIsaac/NewRepoML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

I tested staleness and visibility first. Staleness was `OPPOSITE` in this slice, so I will not use it in the rule. Visibility was `CONFIRMED`; I will pair it with low CTR among pages that have a usable search position. A page is highest priority when it has at least 500 impressions in 90 days, ranks at position 20 or better, and has CTR below 0.5%. This is a fixed human-readable CTR-fix rule, not a fitted model.

Each page receives exactly one reason code:

- `visible_low_ctr`: enough demand, usable position, and unusually low CTR; review the title/snippet first.
- `visible_other`: visible but not in the low-CTR opportunity bucket; monitor.
- `low_visibility`: too little measured demand for a confident CTR intervention.

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_dataset() -> Path:
    candidates = [
        Path.cwd() / "data" / "raw" / "content_refresh_anonymized.csv",
        Path.cwd().parent / "data" / "raw" / "content_refresh_anonymized.csv",
        Path.cwd().parent.parent / "data" / "raw" / "content_refresh_anonymized.csv",
        Path("data/raw/content_refresh_anonymized.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find the starter dataset.")


df = pd.read_csv(find_dataset())
df["is_declining_label"] = df["trend_direction"].fillna("").str.lower().eq("down").astype(int)
df["visible_flag"] = (df["impressions_90d"] >= 500).astype(int)
df["usable_position_flag"] = (df["avg_position"] > 0).astype(int)
df["page_one_or_two_flag"] = ((df["avg_position"] > 0) & (df["avg_position"] <= 20)).astype(int)
df["low_ctr_flag"] = (df["ctr"] < 0.5).astype(int)


def signal_bucket_table(frame, column, bins, labels):
    bucket = pd.cut(frame[column], bins=bins, labels=labels, include_lowest=True)
    return frame.assign(bucket=bucket).groupby("bucket", observed=False).agg(
        n=("content_id", "size"),
        declining_rate=("is_declining_label", "mean"),
        median_impressions=("impressions_90d", "median"),
    ).reset_index()


def verdict_for_high_bucket(table):
    rates = table["declining_rate"].dropna()
    if len(rates) < 2:
        return "FALSE"
    difference = rates.iloc[-1] - rates.iloc[0]
    if abs(difference) < 0.03:
        return "MIXED"
    return "CONFIRMED" if difference > 0 else "OPPOSITE"


visibility_table = signal_bucket_table(
    df, "impressions_90d", [-np.inf, 99, 499, 1999, np.inf],
    ["0-99", "100-499", "500-1999", "2000+"],
)
position_table = signal_bucket_table(
    df.loc[df["avg_position"] > 0].copy(), "avg_position",
    [-np.inf, 3, 10, 20, np.inf], ["1-3", "4-10", "11-20", "20+"],
)

print("Rows loaded:", len(df))
print("Signal check 1: impressions_90d")
print("n:", int(visibility_table["n"].sum()))
display(visibility_table)
print("Verdict:", verdict_for_high_bucket(visibility_table))
print("Signal check 2: avg_position (usable positions only)")
print("n:", int(position_table["n"].sum()))
display(position_table)
print("Verdict:", verdict_for_high_bucket(position_table))

Rows loaded: 30000
Signal check 1: impressions_90d
n: 30000


,bucket,n,declining_rate,median_impressions
0,0-99,7994,0.389042,12.0
1,100-499,5280,0.604356,251.5
2,500-1999,6511,0.617570,1009.0
3,2000+,10215,0.581498,6473.0


Verdict: CONFIRMED
Signal check 2: avg_position (usable positions only)
n: 28795


,bucket,n,declining_rate,median_impressions
0,1-3,1141,0.497809,74.0
1,4-10,11842,0.569414,1184.0
2,11-20,7273,0.609515,870.0
3,20+,8539,0.528165,648.0


Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

The frozen rule below writes one row per page to `work/outputs/baseline_action_score.csv`.

In [7]:
# One frozen rule: visible pages in positions 1-20 with low CTR receive the highest priority.
df["baseline_action_score"] = (
    2 * (df["visible_flag"] & df["page_one_or_two_flag"] & df["low_ctr_flag"]).astype(int)
    + df["visible_flag"]
)


def reason_code(row):
    if row["visible_flag"] and row["page_one_or_two_flag"] and row["low_ctr_flag"]:
        return "visible_low_ctr"
    if row["visible_flag"]:
        return "visible_other"
    return "low_visibility"


def action_label(reason):
    if reason == "visible_low_ctr":
        return "review_title_and_snippet"
    if reason == "visible_other":
        return "monitor"
    return "monitor_low_demand"


df["reason_code"] = df.apply(reason_code, axis=1)
df["action_label"] = df["reason_code"].map(action_label)
df["baseline_rank"] = df["baseline_action_score"].rank(method="first", ascending=False).astype(int)

queue_columns = [
    "baseline_rank", "content_id", "client_id", "baseline_action_score",
    "reason_code", "action_label", "is_declining_label", "impressions_90d",
    "avg_position", "ctr",
]
queue = df[queue_columns].sort_values("baseline_rank").reset_index(drop=True)
repo_root = find_dataset().parents[2]
output_path = repo_root / "work" / "outputs" / "baseline_action_score.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)

precision_at_50 = queue.head(50)["is_declining_label"].mean()
print("Queue rows:", len(queue))
print("Wrote:", output_path)
print("Precision@50:", round(precision_at_50, 3))
print("Random-selection base rate:", round(df["is_declining_label"].mean(), 3))
display(queue.head(10))

Queue rows: 30000
Wrote: f:\GitHub\NewRepoML\work\outputs\baseline_action_score.csv
Precision@50: 0.68
Random-selection base rate: 0.542


,baseline_rank,content_id,client_id,baseline_action_score,reason_code,action_label,is_declining_label,impressions_90d,avg_position,ctr
0,1,content_331d6c4de07b,client_19581e27de,3,visible_low_ctr,review_title_and_snippet,0,11751,6.2,0.49
1,2,content_d4084a4bc775,client_f369cb89fc,3,visible_low_ctr,review_title_and_snippet,1,3970,8.5,0.03
2,3,content_c27558df2b0c,client_19581e27de,3,visible_low_ctr,review_title_and_snippet,1,1240,4.9,0.16
3,4,content_78bd1d4a1d4d,client_6208ef0f77,3,visible_low_ctr,review_title_and_snippet,1,13848,8.9,0.15
4,5,content_761a44afda12,client_19581e27de,3,visible_low_ctr,review_title_and_snippet,1,9449,7.3,0.07
5,6,content_0b360eb9db55,client_349c41201b,3,visible_low_ctr,review_title_and_snippet,1,5141,11.4,0.14
6,7,content_3fb46bec4413,client_4e07408562,3,visible_low_ctr,review_title_and_snippet,1,2311,3.9,0.13
7,8,content_0e23e310d404,client_19581e27de,3,visible_low_ctr,review_title_and_snippet,1,29541,4.1,0.27
8,9,content_7ea135180dd9,client_4ec9599fc2,3,visible_low_ctr,review_title_and_snippet,1,1197,8.1,0.17
9,10,content_55f75c034970,client_d029fa3a95,3,visible_low_ctr,review_title_and_snippet,0,3998,6.4,0.03


## 3. Top-10 review

The top ten are reviewed as decision-support candidates. Each line reports the proposed action, the rule's reason, and the evidence that could make the recommendation wrong, such as noisy low-volume rates or an update that the metadata failed to record.

In [8]:
top10 = queue.head(10).copy()
top10["review_line"] = top10.apply(
    lambda row: (
        f"{row['baseline_rank']}. action={row['action_label']}; "
        f"why={row['reason_code']} (score={row['baseline_action_score']}); "
        f"wrong_if=CTR is noisy, the position is unavailable or outside 1-20, or the snippet already matches the query intent."
    ),
    axis=1,
)

for line in top10["review_line"]:
    print(line)

assert len(top10) == 10
assert top10["reason_code"].notna().all()
assert top10["action_label"].notna().all()
assert (top10["reason_code"] == "visible_low_ctr").all()

1. action=review_title_and_snippet; why=visible_low_ctr (score=3); wrong_if=CTR is noisy, the position is unavailable or outside 1-20, or the snippet already matches the query intent.
2. action=review_title_and_snippet; why=visible_low_ctr (score=3); wrong_if=CTR is noisy, the position is unavailable or outside 1-20, or the snippet already matches the query intent.
3. action=review_title_and_snippet; why=visible_low_ctr (score=3); wrong_if=CTR is noisy, the position is unavailable or outside 1-20, or the snippet already matches the query intent.
4. action=review_title_and_snippet; why=visible_low_ctr (score=3); wrong_if=CTR is noisy, the position is unavailable or outside 1-20, or the snippet already matches the query intent.
5. action=review_title_and_snippet; why=visible_low_ctr (score=3); wrong_if=CTR is noisy, the position is unavailable or outside 1-20, or the snippet already matches the query intent.
6. action=review_title_and_snippet; why=visible_low_ctr (score=3); wrong_if=CTR 

## 4. Weak picks + leakage check

The rule is intentionally simple, so a weak pick is expected. A low CTR can be caused by small samples, query intent mismatch, or a snippet that is already appropriate. The audit below identifies top-10 candidates with weak evidence and verifies that label-source fields and product flags were not used to calculate the score.

In [9]:
weak_picks = queue.head(10).query("impressions_90d < 1000")
if weak_picks.empty:
    weak_picks = queue.head(10).query("ctr < 0.2")

print("Weak picks found in top 10:", len(weak_picks))
if weak_picks.empty:
    print("No weak-pick category appeared in the top 10; inspect the boundary manually.")
else:
    display(weak_picks[["baseline_rank", "reason_code", "action_label", "impressions_90d", "avg_position", "ctr"]])

score_inputs = {"visible_flag", "page_one_or_two_flag", "low_ctr_flag"}
for forbidden in ["trend_direction", "trend_pct", "is_declining_label", "health_score", "priority_score", "action_type"]:
    assert forbidden not in score_inputs

assert queue["reason_code"].map(lambda value: isinstance(value, str)).all()
assert output_path == find_dataset().parents[2] / "work" / "outputs" / "baseline_action_score.csv"
assert output_path.exists()
print("Leakage check passed: score uses only visible_flag, page_one_or_two_flag, and low_ctr_flag.")

Weak picks found in top 10: 8


,baseline_rank,reason_code,action_label,impressions_90d,avg_position,ctr
1,2,visible_low_ctr,review_title_and_snippet,3970,8.5,0.03
2,3,visible_low_ctr,review_title_and_snippet,1240,4.9,0.16
3,4,visible_low_ctr,review_title_and_snippet,13848,8.9,0.15
4,5,visible_low_ctr,review_title_and_snippet,9449,7.3,0.07
5,6,visible_low_ctr,review_title_and_snippet,5141,11.4,0.14
6,7,visible_low_ctr,review_title_and_snippet,2311,3.9,0.13
8,9,visible_low_ctr,review_title_and_snippet,1197,8.1,0.17
9,10,visible_low_ctr,review_title_and_snippet,3998,6.4,0.03


Leakage check passed: score uses only visible_flag, page_one_or_two_flag, and low_ctr_flag.


## Self-check

- [x] Two signals checked with bucket tables, printed `n`, and one-word verdicts.
- [x] One transparent rule produces one score, one reason code, one action label, and a ranked CSV at `work/outputs/baseline_action_score.csv`.
- [x] Top-10 review reports action, reason, and what could make each pick wrong.
- [x] Weak picks and leakage audit completed.
- [ ] Notebook run top to bottom and committed under `work/notebooks/`.